# Effect of Antisemitic Tropes on Recall 

In [1]:
import pandas as pd
import numpy as np
import os
import sys
from os.path import join, abspath

parent_dir = abspath(os.path.join(os.getcwd(), '../..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from config import DATA_DIR
from utils.classification_helpers import group_ihra_content, group_lexicon_content
from utils.stats_helpers import pairwise_segment_tests
from utils.analysis_helpers import calculate_recall, summarize_recall_stats, group_dfs_by_row_index_pivot, explode_df, test_recall_per_content_group_compared_to_base, combine_dataset_tests

pd.set_option("display.max_rows", 100)

PROVIDER = "claude"

In [2]:
bloomington_tmp = pd.read_feather(join(DATA_DIR, f"{PROVIDER}_bloomington_label_1.feather"))
decoding = pd.read_feather(join(DATA_DIR, f"{PROVIDER}_decoding_label_1.feather"))

In [3]:
print("Bloomington columns:", bloomington_tmp.columns)

Bloomington columns: Index(['comment_cleaned', 'id', 'keyword', 'ihra_section_1', 'ihra_section_2',
       'classification_tax', 'explanation_tax', 'classification_tax_ex',
       'explanation_tax_ex', 'classification_ihra_explanation_cleaned',
       'explanation_ihra_explanation_cleaned', 'classification_no_kb_cleaned',
       'explanation_ihra_explanation_sections', 'explanation_tax_chapters',
       'explanation_tax_ex_chapters', 'ihra_sections',
       'explanation_tax_chapters_no', 'explanation_tax_ex_chapters_no',
       'explanation_tax_sections', 'explanation_tax_ex_sections'],
      dtype='object')


In [4]:
column_name_renaming = {
    'classification_no_kb_cleaned': 'NO_KB',
    'classification_ihra_explanation_cleaned': 'IHRA',
    'classification_tax': 'TAX',
    'classification_tax_ex': 'TAX_EX',
}

bloomington_tmp.rename(columns=column_name_renaming, inplace=True)
decoding.rename(columns=column_name_renaming, inplace=True)

REL_CLASS_COLS = column_name_renaming.values()

In [5]:
# Create a new df for Bloomington data with a new column "ihra_sec_new" containing both annotators' decisions.
bloomington_copy_x = bloomington_tmp.copy()
bloomington_copy_y = bloomington_tmp.copy()
bloomington_copy_x["ihra_sec_new"] = bloomington_tmp["ihra_section_1"]
bloomington_copy_y["ihra_sec_new"] = bloomington_tmp["ihra_section_2"]
bloomington = pd.concat([bloomington_copy_x, bloomington_copy_y], ignore_index=True)
print(len(bloomington))
bloomington = bloomington[bloomington["ihra_sec_new"]!=13] # there are a few errors coded as 13
print(len(bloomington))

3716
3635


## Bloomington

#### a. Within each KB

In [6]:

class_cols_to_recalls = {}
for classification_column in REL_CLASS_COLS:
    recall,support, correct = calculate_recall(df=bloomington, classification_column=classification_column, split_by='keyword', two_annotators=True)
    recall_summary = summarize_recall_stats(recall, support, correct)
    print(f"------{classification_column}-------")
    print(recall_summary)
    class_cols_to_recalls[classification_column]=recall_summary
    pairwise = pairwise_segment_tests(recall_summary, correction="holm")
    print("\n[Pairwise tests vs each other] (Holm-corrected)")
    print(pairwise[["seg_A","seg_B","p_raw","p_adj","significant", "effect_h"]])
    print("---------\n")    


------NO_KB-------
         Recall  Support  Correct  Missed
Israel     0.19      641      124     517
Kikes      0.98       89       88       1
ZioNazi    0.95      397      377      20
Jews       0.60      689      415     274

[Pairwise tests vs each other] (Holm-corrected)
     seg_A    seg_B          p_raw          p_adj  significant  effect_h
1   Israel  ZioNazi  1.889817e-123  1.133890e-122         True -1.788512
0   Israel    Kikes   2.884829e-53   1.442415e-52         True -1.955745
2   Israel     Jews   1.184267e-51   4.737069e-51         True -0.870101
5  ZioNazi     Jews   5.953109e-35   1.785933e-34         True  0.918412
4    Kikes     Jews   1.677704e-12   3.355407e-12         True  1.085644
3    Kikes  ZioNazi   1.465800e-01   1.465800e-01        False  0.167233
---------

------IHRA-------
         Recall  Support  Correct  Missed
Israel     0.43      641      276     365
Kikes      1.00       89       89       0
ZioNazi    1.00      397      396       1
Jews       0.6

#### b. Between KBs

In [7]:
grouped = group_dfs_by_row_index_pivot(class_cols_to_recalls)
print(grouped["Kikes"])

         Correct  Missed  Recall  Support
dataset                                  
IHRA          89       0    1.00       89
NO_KB         88       1    0.98       89
TAX           89       0    1.00       89
TAX_EX        89       0    1.00       89


In [8]:
print(grouped["Israel"])

         Correct  Missed  Recall  Support
dataset                                  
IHRA         276     365    0.43      641
NO_KB        124     517    0.19      641
TAX          446     195    0.70      641
TAX_EX       339     302    0.53      641


### Evaluation by content groups

Content groups allow to compare IHRA sections with LEXICON chapters

In [10]:
bloomington["ihra_content"] = bloomington["ihra_sec_new"].map(group_ihra_content)
print(bloomington["ihra_content"].value_counts(dropna=False, normalize=True))
print(type(bloomington["ihra_content"][0]))

ihra_content
israel                  0.576891
classic_power           0.279230
aggressive              0.096286
second_postholocaust    0.047593
Name: proportion, dtype: float64
<class 'str'>


#### a. Within each KB

In [11]:
for classification_column in REL_CLASS_COLS:
    recall,support, correct = calculate_recall(df=bloomington, classification_column=classification_column, split_by='ihra_content', two_annotators=True)
    recall_summary = summarize_recall_stats(recall, support, correct)
    print(f"--------{classification_column}--------")
    print(recall_summary)
    class_cols_to_recalls[classification_column]=recall_summary
    pairwise = pairwise_segment_tests(recall_summary, correction="holm")
    print(pairwise[["seg_A","seg_B","p_raw","p_adj","significant", "effect_h"]])
    print("---------\n")    

--------NO_KB--------
                      Recall  Support  Correct  Missed
israel                  0.41     1048      427     621
classic_power           0.78      507      393     114
aggressive              0.82      175      143      32
second_postholocaust    0.49       86       42      44
           seg_A                 seg_B         p_raw         p_adj  \
0         israel         classic_power  6.891434e-42  4.134861e-41   
1         israel            aggressive  1.950519e-23  9.752595e-23   
4  classic_power  second_postholocaust  5.624411e-08  2.249764e-07   
5     aggressive  second_postholocaust  8.779102e-08  2.633731e-07   
2         israel  second_postholocaust  1.766501e-01  3.533002e-01   
3  classic_power            aggressive  2.887414e-01  3.533002e-01   

   significant  effect_h  
0         True -0.775372  
1         True -0.875485  
4         True  0.614387  
5         True  0.714500  
2        False -0.160985  
3        False -0.100112  
---------

--------IHRA

In [12]:
# NEW!
kb_to_recall_b = {}
for kb in REL_CLASS_COLS:
    kb_to_recall_b[kb] = class_cols_to_recalls[kb]['Recall'].to_dict()
pd.DataFrame(kb_to_recall_b)

,NO_KB,IHRA,TAX,TAX_EX
israel,0.41,0.58,0.75,0.64
classic_power,0.78,0.83,0.86,0.83
aggressive,0.82,0.86,0.87,0.85
second_postholocaust,0.49,0.62,0.67,0.59


#### b. Between KBs

In [13]:
test_recall_per_content_group_compared_to_base(class_cols_to_recalls, 0)

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
content_group,,,,,,
aggressive,NO_KB,IHRA,0.385072,1.0,False,-0.109304
classic_power,NO_KB,IHRA,0.0331,0.165502,False,-0.126433
israel,NO_KB,IHRA,0.0,0.0,True,-0.341677
second_postholocaust,NO_KB,IHRA,0.125176,0.62588,False,-0.262367


In [14]:
test_recall_per_content_group_compared_to_base(class_cols_to_recalls, 3)

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
content_group,,,,,,
aggressive,NO_KB,TAX,0.182931,1.0,False,-0.138572
classic_power,NO_KB,TAX,0.000459,0.002752,True,-0.209417
israel,NO_KB,TAX,0.0,0.0,True,-0.704585
second_postholocaust,NO_KB,TAX,0.020427,0.122564,False,-0.366918


In [15]:
test_recall_per_content_group_compared_to_base(class_cols_to_recalls, 4)

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
content_group,,,,,,
aggressive,NO_KB,TAX_EX,0.567924,1.0,False,-0.080899
classic_power,NO_KB,TAX_EX,0.04934,0.197358,False,-0.126433
israel,NO_KB,TAX_EX,0.0,0.0,True,-0.464781
second_postholocaust,NO_KB,TAX_EX,0.220933,0.883733,False,-0.200988


In [16]:
# NEW: combine all tests in one dataframe for aggregation between datasets
recall_tests_b = pd.concat([test_recall_per_content_group_compared_to_base(class_cols_to_recalls, i) for i in [0,3,4]])
recall_tests_b

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
content_group,,,,,,
aggressive,NO_KB,IHRA,0.385072,1.0,False,-0.109304
classic_power,NO_KB,IHRA,0.0331,0.165502,False,-0.126433
israel,NO_KB,IHRA,0.0,0.0,True,-0.341677
second_postholocaust,NO_KB,IHRA,0.125176,0.62588,False,-0.262367
aggressive,NO_KB,TAX,0.182931,1.0,False,-0.138572
classic_power,NO_KB,TAX,0.000459,0.002752,True,-0.209417
israel,NO_KB,TAX,0.0,0.0,True,-0.704585
second_postholocaust,NO_KB,TAX,0.020427,0.122564,False,-0.366918
aggressive,NO_KB,TAX_EX,0.567924,1.0,False,-0.080899


## Decoding

### Evaluation by content groups



In [17]:
decoding["lexicon_chapter_groups"] = decoding["comment_codes_all_chapters"].map(group_lexicon_content)

In [18]:
# restrict to texts with maximum two content groups assigned
decoding = decoding[decoding['lexicon_chapter_groups'].map(lambda x: len(x) <=2 and len(x)>0)].copy()
decoding["lexicon_chapter_groups"] = decoding["lexicon_chapter_groups"].map(lambda x: x if len(x)==2 else x*2)
decoding = explode_df(decoding, "lexicon_chapter_groups")

In [19]:
decoding["lexicon_chapter_groups"].value_counts()

lexicon_chapter_groups
israel                  2667
classic_power           2169
second_postholocaust     775
aggressive               131
Name: count, dtype: int64

#### a. Within each KB

In [20]:
for classification_column in REL_CLASS_COLS:
    recall,support, correct = calculate_recall(df=decoding, classification_column=classification_column, split_by='lexicon_chapter_groups', two_annotators=True)
    recall_summary = summarize_recall_stats(recall, support, correct)
    print(f"--------{classification_column}--------")
    print(recall_summary)
    class_cols_to_recalls[classification_column]=recall_summary
    pairwise = pairwise_segment_tests(recall_summary, correction="holm")
    print("\n[Pairwise tests vs each other] (Holm-corrected)")
    print(pairwise[["seg_A","seg_B","p_raw","p_adj","significant", "effect_h"]])
    print("---------\n")   

--------NO_KB--------
                      Recall  Support  Correct  Missed
israel                  0.27     1333      359     974
classic_power           0.38     1084      409     675
second_postholocaust    0.29      387      110     277
aggressive              0.65       65       42      23

[Pairwise tests vs each other] (Holm-corrected)
                  seg_A                 seg_B         p_raw         p_adj  \
2                israel            aggressive  1.372691e-10  8.236148e-10   
0                israel         classic_power  1.833310e-08  9.166552e-08   
5  second_postholocaust            aggressive  2.503760e-08  1.001504e-07   
4         classic_power            aggressive  2.906451e-05  8.719352e-05   
3         classic_power  second_postholocaust  1.250341e-03  2.500681e-03   
1                israel  second_postholocaust  6.062737e-01  6.062737e-01   

   significant  effect_h  
2         True -0.782688  
0         True -0.235629  
5         True -0.738138  
4     

In [21]:
kb_to_recall_d = {}
for kb in REL_CLASS_COLS:
    kb_to_recall_d[kb] = class_cols_to_recalls[kb]['Recall'].to_dict()
pd.DataFrame(kb_to_recall_d)

,NO_KB,IHRA,TAX,TAX_EX
israel,0.27,0.49,0.71,0.55
classic_power,0.38,0.47,0.65,0.53
second_postholocaust,0.29,0.37,0.55,0.47
aggressive,0.65,0.76,0.84,0.80


#### b. Between KBs

In [22]:
test_recall_per_content_group_compared_to_base(class_cols_to_recalls, 0)

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
content_group,,,,,,
aggressive,NO_KB,IHRA,0.250829,1.0,False,-0.242158
classic_power,NO_KB,IHRA,0.00002,0.000041,True,-0.18233
israel,NO_KB,IHRA,0.0,0.0,True,-0.457994
second_postholocaust,NO_KB,IHRA,0.014202,0.028404,True,-0.170423


In [23]:
test_recall_per_content_group_compared_to_base(class_cols_to_recalls, 3)

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
content_group,,,,,,
aggressive,NO_KB,TAX,0.015593,0.093559,False,-0.44307
classic_power,NO_KB,TAX,0.0,0.0,True,-0.547059
israel,NO_KB,TAX,0.0,0.0,True,-0.911441
second_postholocaust,NO_KB,TAX,0.0,0.0,True,-0.533613


In [24]:
test_recall_per_content_group_compared_to_base(class_cols_to_recalls, 4)

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
content_group,,,,,,
aggressive,NO_KB,TAX_EX,0.077732,0.388658,False,-0.338808
classic_power,NO_KB,TAX_EX,0.0,0.0,True,-0.302402
israel,NO_KB,TAX_EX,0.0,0.0,True,-0.578163
second_postholocaust,NO_KB,TAX_EX,0.0,0.000001,True,-0.373409


In [25]:
recall_tests_d = pd.concat([test_recall_per_content_group_compared_to_base(class_cols_to_recalls, i) for i in [0,3,4]])
recall_tests_d

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
content_group,,,,,,
aggressive,NO_KB,IHRA,0.250829,1.0,False,-0.242158
classic_power,NO_KB,IHRA,0.00002,0.000041,True,-0.18233
israel,NO_KB,IHRA,0.0,0.0,True,-0.457994
second_postholocaust,NO_KB,IHRA,0.014202,0.028404,True,-0.170423
aggressive,NO_KB,TAX,0.015593,0.093559,False,-0.44307
classic_power,NO_KB,TAX,0.0,0.0,True,-0.547059
israel,NO_KB,TAX,0.0,0.0,True,-0.911441
second_postholocaust,NO_KB,TAX,0.0,0.0,True,-0.533613
aggressive,NO_KB,TAX_EX,0.077732,0.388658,False,-0.338808


## Dataset union 

- equal dataset weight
- by content groups

In [26]:
# combine recalls per content group
# also need to fetch overall recall first

In [27]:
kb_to_recall_bd = {}
for classification_column in REL_CLASS_COLS:
    tmp_b = kb_to_recall_b[classification_column]
    tmp_d = kb_to_recall_d[classification_column]
    tmp = {k:0.5*(tmp_b[k] + tmp_d[k]) for k in tmp_b.keys()}
    kb_to_recall_bd[classification_column] = tmp
       

In [28]:
kb_to_recall_bd = pd.DataFrame(kb_to_recall_bd)
kb_to_recall_bd

,NO_KB,IHRA,TAX,TAX_EX
israel,0.340,0.535,0.730,0.595
classic_power,0.580,0.650,0.755,0.680
aggressive,0.735,0.810,0.855,0.825
second_postholocaust,0.390,0.495,0.610,0.530


In [29]:
kb_to_recall_bd.loc["mean",:] = kb_to_recall_bd.mean().tolist()

In [30]:
kb_to_recall_bd

,NO_KB,IHRA,TAX,TAX_EX
israel,0.34000,0.5350,0.7300,0.5950
classic_power,0.58000,0.6500,0.7550,0.6800
aggressive,0.73500,0.8100,0.8550,0.8250
second_postholocaust,0.39000,0.4950,0.6100,0.5300
mean,0.51125,0.6225,0.7375,0.6575


In [29]:
kb_to_recall_bd.to_parquet(f"{PROVIDER}_recall_by_trope.parquet", index=True)

In [31]:
combined = combine_dataset_tests(
    recall_tests_b,
    recall_tests_d,
)
combined

,content_group,seg_A,seg_B,p_raw,effect_h,p_adj,significant
6,israel,NO_KB,TAX,7.274069e-30,-0.808013,8.728883e-29,True
10,israel,NO_KB,TAX_EX,7.274069e-30,-0.521472,8.728883e-29,True
2,israel,NO_KB,IHRA,1.340040e-29,-0.399835,1.340040e-28,True
5,classic_power,NO_KB,TAX,3.538093e-16,-0.378238,3.184284e-15,True
7,second_postholocaust,NO_KB,TAX,3.759746e-12,-0.450265,3.007797e-11,True
9,classic_power,NO_KB,TAX_EX,2.926739e-10,-0.214417,2.048717e-09,True
11,second_postholocaust,NO_KB,TAX_EX,4.462622e-06,-0.287199,2.677573e-05,True
1,classic_power,NO_KB,IHRA,6.205906e-06,-0.154381,3.102953e-05,True
3,second_postholocaust,NO_KB,IHRA,4.829512e-03,-0.216395,1.931805e-02,True
4,aggressive,NO_KB,TAX,8.008542e-03,-0.290821,2.402563e-02,True
